In [8]:
import os
import glob
import pandas as pd
import shutil
from datetime import timedelta
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from datetime import datetime

camera='ffc'
letter=['a','b','c','d','e','f','g','h','j','k']

# Define start and end times as datetime objects
start_time = datetime.strptime('2022-08-07 1736', '%Y-%m-%d %H%M')
end_time = datetime.strptime('2022-08-07 1746', '%Y-%m-%d %H%M')#1736

#start_time = datetime.strptime('2022-07-26 1538', '%Y-%m-%d %H%M') #1503
#end_time = datetime.strptime('2022-07-26 1548', '%Y-%m-%d %H%M') 
shift='yes'
shift_time = 11

#start_time = datetime.strptime('2022-07-26 1906', '%Y-%m-%d %H%M')
#end_time = datetime.strptime('2022-07-26 1916', '%Y-%m-%d %H%M')

#start_time = datetime.strptime('2022-07-27 1726', '%Y-%m-%d %H%M')
#end_time = datetime.strptime('2022-07-27 1736', '%Y-%m-%d %H%M')

#output_folder = glob.glob('output_frames/c303/20220725/' + camera +'/*.png')
#output_folder = glob.glob('output_frames/c304/20220726/' + camera +'/*.png')
#output_folder = glob.glob('output_frames/c301/20220723/' + camera +'/*.png')
output_folder = glob.glob('output_frames/c314/20220807/' + camera +'/*.png')
def create_dataframe_from_images(output_folder):
    # Get a list of all saved image files in the output folder
    #image_files = [file for file in output_folder]
    #Extract information from the image filenames
    data = []
    for image_file in output_folder:
        filename_parts = image_file.split('_')
        flight_id = filename_parts[2]
        date = filename_parts[3]
        times = filename_parts[4][0:-4]
        # Convert timestamp to full datetime format
        full_timestamp = pd.to_datetime(date+'_'+times, format="%Y%m%d_%H%M%S")
        data.append({
            'filename': image_file,
            'flight_id': flight_id,
            'timestamp': full_timestamp
        })

    # Create a DataFrame from the extracted information
    df = pd.DataFrame(data)
    df = df.sort_values(by='timestamp').reset_index(drop=True)

    return df

In [9]:

# Open the netCDF file
#dataset = xr.open_dataset('core_faam_20220726_v005_r0_c304.nc')
dataset = xr.open_dataset('core_faam_20220807_v005_r0_c314_1hz.nc')
#dataset = xr.open_dataset('core_faam_20220725_v005_r0_c303.nc')
#dataset = xr.open_dataset('core_faam_20220801_v005_r0_c309.nc') 
roll =dataset['ROLL_GIN']
roll_times = roll.Time
roll_angle = roll.data[:]
roll_times_pd = pd.to_datetime(roll_times.data)

In [10]:


roll_df = pd.DataFrame({'roll_times': roll_times_pd, 'roll_angles': roll_angle})


In [11]:
frame_times = create_dataframe_from_images(output_folder)

In [12]:
# Define a function to find the closest roll_time and get the corresponding roll_angle
def get_closest_roll_angle(frame_time):
    # Calculate the absolute difference between frame_time and all roll_times
    diffs = abs(roll_df['roll_times'] - frame_time)
    # Find the index of the minimum difference
    min_diff_index = diffs.idxmin()
    # Return the corresponding roll_angle
    return roll_df.loc[min_diff_index, 'roll_angles']

# Define a function to find the closest roll_time and get the corresponding roll_angle
def get_closest_roll_angle_shift(frame_time, shift):
    # Calculate the absolute difference between frame_time and all roll_times
    diffs = abs(roll_df['roll_times'] - frame_time+timedelta(seconds=shift))
    # Find the index of the minimum difference
    min_diff_index = diffs.idxmin()
    # Return the corresponding roll_angle
    return roll_df.loc[min_diff_index, 'roll_angles']

# Apply the function to each row in frame_times
if shift=='no':
    print('no shift')
    frame_times['roll_angles'] = frame_times['timestamp'].apply(get_closest_roll_angle)
elif shift=='yes':
    frame_times['roll_angles'] = frame_times['timestamp'].apply(get_closest_roll_angle_shift, shift=-shift_time)

In [13]:


# Subset the DataFrame
subset_frame_times = frame_times[(frame_times['timestamp'] >= start_time) & (frame_times['timestamp'] <= end_time)]

In [7]:

from PIL import Image

# Create the output directory if it doesn't exist
if not os.path.exists('timelapse'):
    os.makedirs('timelapse')

# Loop through each row in subset_frame_times
for index, row in subset_frame_times.iterrows():
    # Open the image
    # img = mpimg.imread(row['filename'])
    img = Image.open(row['filename'])
    img.thumbnail((800, 800))  # Adjust the size as needed
    img = np.array(img)
    # Create a new figure
    plt.figure()
    
    # Display the image
    plt.imshow(img)
    
    # Display the roll_angle
    plt.text(0.1, 0.8, f'{row['roll_angles']:,.2f}',
             color='red',
             horizontalalignment='left',
             verticalalignment='center',
             transform = plt.gca().transAxes)
    # Calculate line coordinates for roll angle
    if camera == 'ffc':
        angle_rad = np.deg2rad(-row['roll_angles'])
    else:
        angle_rad = np.deg2rad(row['roll_angles'])
    line_length = img.shape[1] / 2
    x = [img.shape[1] / 2 - line_length * np.cos(angle_rad), img.shape[1] / 2 + line_length * np.cos(angle_rad)]
    y = [img.shape[0] / 2 - line_length * np.sin(angle_rad), img.shape[0] / 2 + line_length * np.sin(angle_rad)]
    
    # Plot line
    plt.plot(x, y, color='red')
    # Save the plot
    plt.savefig(f'timelapse/{index}.png')
    plt.close()
    # Show the plot
    

In [413]:
roll_df.dropna()


,roll_times,roll_angles
4913,2022-07-26 14:22:33,0.230370
4914,2022-07-26 14:22:34,0.229916
4915,2022-07-26 14:22:35,0.229480
4916,2022-07-26 14:22:36,0.228500
4917,2022-07-26 14:22:37,0.229298
...,...,...
24139,2022-07-26 19:42:59,0.279731
24140,2022-07-26 19:43:00,0.280730
24141,2022-07-26 19:43:01,0.282114
24142,2022-07-26 19:43:02,0.281713


In [415]:
#dataset = xr.open_dataset('core_faam_20220727_v005_r0_c305_1hz.nc')
alti =dataset['ALT_GIN']
alti_times = alti.Time
alti_height = alti.data
alti_times_pd = pd.to_datetime(roll_times.data)

In [418]:

alti_df = pd.DataFrame({'alti_times': alti_times_pd, 'height': alti_height})

In [419]:
alti_df.dropna()

,alti_times,height
4913,2022-07-26 14:22:33,1599.408203
4914,2022-07-26 14:22:34,1599.408203
4915,2022-07-26 14:22:35,1599.408203
4916,2022-07-26 14:22:36,1599.408203
4917,2022-07-26 14:22:37,1599.408325
...,...,...
24139,2022-07-26 19:42:59,1601.145752
24140,2022-07-26 19:43:00,1601.145508
24141,2022-07-26 19:43:01,1601.145264
24142,2022-07-26 19:43:02,1601.145020
